# Fair APSL vs TAMO on Kaggle

This notebook clones the repository, installs runtime dependencies, creates one shared training dataset, trains TAMO and APSL, and evaluates both on four problems with 10 paired seeds. Every test run uses **100 initial + 100 sequential = 200 function evaluations**.

Shared training settings: 400 epochs, 160 burn-in epochs, batch size 16, the same Transformer width/depth, learning rates, rollout horizon, synthetic pool size, and 64 action candidates. At test time both methods see 128 candidates per step. Architectural differences intrinsic to APSL are intentionally retained.

In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working
if [ -d AmortizedPSL/.git ]; then
    git -C AmortizedPSL pull --ff-only
else
    git clone https://github.com/phamduynguyenlam/AmortizedPSL.git
fi

cd AmortizedPSL
git rev-parse --short HEAD


In [ ]:
import subprocess
import sys

packages = [
    'torchdata==0.11.0',
    'torchrl==0.11.1',
    'gpytorch==1.14',
    'botorch==0.13.0',
    'pyro-ppl==1.9.1',
    'pymoo==0.6.1.6',
    'einops==0.8.2',
    'hydra-core==1.3.2',
    'omegaconf==2.3.0',
    'wandb>=0.19',
    'sobol-seq==0.2.0',
    'flatten-dict==0.4.2',
    'python-dotenv>=1.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])


In [ ]:
from pathlib import Path
import os
import sys
import torch

REPO = Path('/kaggle/working/AmortizedPSL')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

assert torch.cuda.is_available(), 'Enable a GPU accelerator in Kaggle settings.'
print('Repository:', REPO)
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))


## Generate the shared training dataset

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/AmortizedPSL

DATASET=datasets/train/fair_apsl_tamo_v2/x_dim_2/y_dim_2/gp_fair.hdf5
if [ ! -f "$DATASET" ]; then
    python generate_data.py \
        experiment.mode=train \
        experiment.device=cuda \
        experiment.resume=false \
        data.data_id=fair_apsl_tamo_v2 \
        generate.filename=gp_fair \
        generate.x_dim=2 \
        generate.y_dim=2 \
        generate.sampler_type=gp \
        generate.num_datasets=1024 \
        generate.num_datapoints=192
else
    echo "Dataset already exists: $DATASET"
fi
ls -lh "$DATASET"


## Train TAMO

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/AmortizedPSL

RESUME=false
if [ -f results/TAMO/fair100_tamo_v2/ckpt.tar ]; then RESUME=true; fi

time python train.py --config-name=train \
    experiment.seed=0 \
    experiment.device=cuda \
    experiment.model_name=TAMO \
    experiment.expid=fair100_tamo_v2 \
    experiment.resume=$RESUME \
    experiment.log_to_wandb=false \
    data.data_id=fair_apsl_tamo_v2 \
    'data.x_dim_list=[2]' \
    'data.y_dim_list=[2]' \
    prediction.batch_size=16 \
    prediction.min_nc=2 \
    prediction.max_nc=48 \
    train.num_total_epochs=400 \
    train.num_burnin_epochs=160 \
    train.nc_burnin_ratio=0.7 \
    train.num_workers=0 \
    optimization.batch_size=2 \
    optimization.num_samples=1 \
    optimization.num_query_points=64 \
    optimization.num_initial_points=8 \
    optimization.T=8 \
    optimization.min_T=8 \
    optimization.max_T=8 \
    model.max_x_dim=4 \
    model.max_y_dim=3 \
    model.dim_mlp=64 \
    model.dim_attn=64 \
    model.nhead=4 \
    model.dim_hidden=128 \
    model.num_layers_backbone=3 \
    model.num_layers_encoder=3 \
    model.num_layers_decoder=3 \
    model.depth=3 \
    model.num_components=10 \
    loss.loss_weight=1.0 \
    log.freq_log=10 \
    log.freq_save=50


## Train APSL

APSL splits the auxiliary weight used by TAMO prediction into `0.5 objective prediction + 0.5 APSL`, while both methods retain REINFORCE weight 1.0. The obsolete scalar prediction loss is disabled for APSL.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/AmortizedPSL

RESUME=false
if [ -f results/APSL/fair100_apsl_v2/ckpt.tar ]; then RESUME=true; fi

time python train.py --config-name=train_apsl \
    experiment.seed=0 \
    experiment.device=cuda \
    experiment.model_name=APSL \
    experiment.expid=fair100_apsl_v2 \
    experiment.resume=$RESUME \
    experiment.log_to_wandb=false \
    data.data_id=fair_apsl_tamo_v2 \
    'data.x_dim_list=[2]' \
    'data.y_dim_list=[2]' \
    prediction.batch_size=16 \
    prediction.min_nc=2 \
    prediction.max_nc=48 \
    train.num_total_epochs=400 \
    train.num_burnin_epochs=160 \
    train.nc_burnin_ratio=0.7 \
    train.num_workers=0 \
    optimization.batch_size=2 \
    optimization.num_samples=1 \
    optimization.num_query_points=64 \
    optimization.num_initial_points=8 \
    optimization.T=8 \
    optimization.min_T=8 \
    optimization.max_T=8 \
    model.max_x_dim=7 \
    model.max_y_dim=1 \
    model.dim_mlp=64 \
    model.dim_attn=64 \
    model.nhead=4 \
    model.dim_hidden=128 \
    model.num_layers_backbone=3 \
    model.num_layers_encoder=3 \
    model.num_layers_decoder=3 \
    model.depth=3 \
    model.num_components=10 \
    loss.loss_weight=0.0 \
    objective_prediction.loss_weight=0.5 \
    apsl.loss_weight=0.5 \
    loss.policy_loss_weight=1.0 \
    apsl.num_train_preferences=64 \
    apsl.num_policy_preferences=64 \
    log.freq_log=10 \
    log.freq_save=50


## Test both methods: 4 problems × 10 paired seeds × 200 FE

Per-run terminal output is written to `kaggle_fair_logs/fair100_v2`; this keeps the notebook readable. Existing artifacts are skipped, so an interrupted notebook can safely resume this cell.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

REPO = Path('/kaggle/working/AmortizedPSL')
TAMO_EXPID = 'fair100_tamo_v2'
APSL_EXPID = 'fair100_apsl_v2'
SUFFIX = 'fair_100plus100_v2'
PROBLEMS = ['AckleyRastrigin', 'AckleyRosenbrock', 'BraninCurrin', 'dx2_dy2']
SEEDS = list(range(10))
LOG_DIR = REPO / 'kaggle_fair_logs' / 'fair100_v2'
LOG_DIR.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env['HYDRA_FULL_ERROR'] = '1'
env['PYTHONUNBUFFERED'] = '1'

common_test = [
    'experiment.device=cuda',
    'experiment.resume=true',
    'experiment.override=true',
    'experiment.log_to_wandb=false',
    'data.max_x_dim=4',
    'data.max_y_dim=3',
    'optimization.batch_size=1',
    'optimization.num_samples=1',
    'optimization.num_initial_points=100',
    'optimization.epsilon=0.0',
    'optimization.num_query_points=128',
]

tamo_model = [
    'model.max_x_dim=4', 'model.max_y_dim=3',
    'model.dim_mlp=64', 'model.dim_attn=64', 'model.nhead=4',
    'model.dim_hidden=128',
    'model.num_layers_backbone=3',
    'model.num_layers_encoder=3',
    'model.num_layers_decoder=3',
    'model.depth=3', 'model.num_components=10',
    'model.dropout=0.0', 'model.std_min=0.001', 'model.std_max=1.0',
]
apsl_model = [
    'model.max_x_dim=7', 'model.max_y_dim=1',
    *tamo_model[2:],
]

def tamo_artifact(problem, seed):
    root = (REPO / 'results' / 'data' / 'TAMO' / TAMO_EXPID /
            'optimization' / problem / 'ckpt' / SUFFIX / str(seed))
    paths = list(root.rglob('hv.pt')) if root.exists() else []
    return paths[0] if paths else None

def apsl_artifact(problem, seed):
    path = (REPO / 'results' / 'data' / 'APSL' / APSL_EXPID /
            'optimization_apsl' / problem / 'ckpt' / SUFFIX /
            str(seed) / 'apsl_rollout.pt')
    return path if path.is_file() else None

def run_logged(label, command, log_path):
    print(label, flush=True)
    started = time.perf_counter()
    with log_path.open('w', encoding='utf-8') as stream:
        completed = subprocess.run(
            command, cwd=str(REPO), env=env,
            stdout=stream, stderr=subprocess.STDOUT, text=True,
        )
    elapsed = time.perf_counter() - started
    if completed.returncode != 0:
        tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-80:]
        raise RuntimeError(f'{label} failed. Log: {log_path}\n' + '\n'.join(tail))
    print(f'  completed in {elapsed / 60:.2f} min; log={log_path}', flush=True)

for problem in PROBLEMS:
    for seed in SEEDS:
        if tamo_artifact(problem, seed) is None:
            command = [
                sys.executable, '-u', 'test.py', '--config-name=test',
                f'experiment.seed={seed}',
                'experiment.model_name=TAMO',
                f'experiment.expid={TAMO_EXPID}',
                f'data.function_name={problem}',
                f'extra.suffix_segment={SUFFIX}',
                # TAMO interprets T as the inclusive total-cost limit.
                # With 100 initial observations, T=199 gives 100 new FE.
                'optimization.T=199',
                'optimization.min_T=199',
                'optimization.max_T=199',
                *common_test,
                'prediction.read_cache=false',
                'optimization.read_cache=false',
                'optimization.write_cache=false',
                'log.plot_enabled=false',
                *tamo_model,
            ]
            run_logged(
                f'TAMO | {problem} | seed={seed}', command,
                LOG_DIR / f'tamo_{problem}_seed{seed}.log',
            )
        else:
            print(f'Skip existing TAMO | {problem} | seed={seed}')

        if apsl_artifact(problem, seed) is None:
            command = [
                sys.executable, '-u', 'test_apsl.py', '--config-name=test_apsl',
                f'experiment.seed={seed}',
                'experiment.model_name=APSL',
                f'experiment.expid={APSL_EXPID}',
                f'data.function_name={problem}',
                f'extra.suffix_segment={SUFFIX}',
                # APSL interprets T directly as the number of new FE.
                'optimization.T=100',
                'optimization.min_T=100',
                'optimization.max_T=100',
                *common_test,
                'apsl.num_policy_preferences=128',
                'apsl.log_every=20',
                *apsl_model,
            ]
            run_logged(
                f'APSL | {problem} | seed={seed}', command,
                LOG_DIR / f'apsl_{problem}_seed{seed}.log',
            )
        else:
            print(f'Skip existing APSL | {problem} | seed={seed}')

print('Finished all paired evaluations.')


## Plot HV regret and create the final table

We calculate the same interpretable metric for both methods directly from saved hypervolume: `HV regret = 1 - HV / max_HV` (lower is better). This avoids the repository's signed internal regret representation.

In [ ]:
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from data.dataset import get_function_environment

N_INITIAL = 100
curves = defaultdict(list)
rows = []

def load_tensor(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(path, map_location='cpu')

def one_dimensional_hv(value):
    array = torch.as_tensor(value).detach().cpu().numpy().squeeze()
    if array.ndim != 1:
        array = array.reshape(-1, array.shape[-1]).mean(axis=0)
    return array.astype(np.float64)

for problem in PROBLEMS:
    for seed in SEEDS:
        environment = get_function_environment(
            function_name=problem, mode='test', seed=seed, device='cpu',
            data_id=None, scene='ship',
        )
        max_hv = float(torch.as_tensor(environment.max_hv).mean())

        tamo_path = tamo_artifact(problem, seed)
        apsl_path = apsl_artifact(problem, seed)
        if tamo_path is None or apsl_path is None:
            raise FileNotFoundError(f'Missing paired artifact for {problem}, seed={seed}')

        tamo_hv = one_dimensional_hv(load_tensor(tamo_path))
        apsl_payload = load_tensor(apsl_path)
        apsl_hv = one_dimensional_hv(apsl_payload['hv'])

        for method, hv in [('TAMO', tamo_hv), ('APSL', apsl_hv)]:
            regret = np.maximum(1.0 - hv / max(max_hv, 1e-12), 0.0)
            curves[(problem, method)].append(regret)
            rows.append({
                'Problem': problem,
                'Method': method,
                'Seed': seed,
                'Initial HV regret': regret[0],
                'Final HV regret': regret[-1],
                'Regret reduction': regret[0] - regret[-1],
                'Final HV / Max HV': hv[-1] / max(max_hv, 1e-12),
                'Evaluations': N_INITIAL + len(regret) - 1,
            })

raw_df = pd.DataFrame(rows).sort_values(['Problem', 'Seed', 'Method'])

fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
for ax, problem in zip(axes.flat, PROBLEMS):
    for method, color in [('TAMO', 'tab:blue'), ('APSL', 'tab:orange')]:
        method_curves = curves[(problem, method)]
        common_length = min(map(len, method_curves))
        values = np.stack([curve[:common_length] for curve in method_curves])
        mean = values.mean(axis=0)
        std = values.std(axis=0, ddof=0)
        evaluations = N_INITIAL + np.arange(common_length)
        ax.plot(evaluations, mean, label=method, color=color, linewidth=2)
        ax.fill_between(evaluations, mean - std, mean + std, color=color, alpha=0.18)
    ax.set_title(problem)
    ax.set_xlabel('Function evaluations')
    ax.set_ylabel('HV regret (lower is better)')
    ax.grid(alpha=0.25)
    ax.legend()

plot_path = REPO / 'results' / 'apsl_vs_tamo_hv_regret_100plus100.png'
fig.savefig(plot_path, dpi=180, bbox_inches='tight')
plt.show()

grouped = (raw_df.groupby(['Problem', 'Method'])['Final HV regret']
           .agg(['mean', 'std', 'count']).reset_index())
paired = raw_df.pivot(index=['Problem', 'Seed'], columns='Method', values='Final HV regret').reset_index()
paired['APSL - TAMO'] = paired['APSL'] - paired['TAMO']
paired_summary = paired.groupby('Problem').agg(
    delta_mean=('APSL - TAMO', 'mean'),
    delta_std=('APSL - TAMO', 'std'),
    apsl_wins=('APSL - TAMO', lambda values: int((values < 0).sum())),
).reset_index()

def formatted(method):
    subset = grouped[grouped['Method'] == method].set_index('Problem')
    return subset.apply(lambda row: f"{row['mean']:.4f} ± {row['std']:.4f}", axis=1)

final_table = pd.DataFrame(index=PROBLEMS)
final_table.index.name = 'Problem'
final_table['TAMO HV regret ↓'] = formatted('TAMO')
final_table['APSL HV regret ↓'] = formatted('APSL')
pair_stats = paired_summary.set_index('Problem')
final_table['APSL − TAMO ↓'] = pair_stats.apply(
    lambda row: f"{row['delta_mean']:.4f} ± {row['delta_std']:.4f}", axis=1
)
final_table['APSL wins'] = pair_stats['apsl_wins'].astype(int).astype(str) + '/10'
final_table = final_table.reset_index()

print('Every row uses 10 paired seeds and 100+100 = 200 FE.')
display(final_table)

raw_path = REPO / 'results' / 'apsl_vs_tamo_raw_100plus100.csv'
table_path = REPO / 'results' / 'apsl_vs_tamo_table_100plus100.csv'
raw_df.to_csv(raw_path, index=False)
final_table.to_csv(table_path, index=False)
print('Saved plot:', plot_path)
print('Saved raw results:', raw_path)
print('Saved table:', table_path)
